# PeMS 站点坐标修正（基于 Abs_PM 匹配）

## 原理
- PeMS 站点的 `Abs_PM` 是可靠的
- Caltrans 官方数据的 `Odometer` 等于 `Abs_PM`
- 用 Caltrans 的坐标替换 PeMS 的坐标

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import glob
import folium
from math import radians, sin, cos, sqrt, atan2
from tqdm import tqdm

# ============== 配置 ==============

# Caltrans GeoJSON 文件路径
CALTRANS_GEOJSON = "/data/yuzhang_fei/PEMS/SHN_Postmiles_Tenth.geojson"

# PeMS 元数据目录
META_DIR = "/data/yuzhang_fei/PEMS/meta"

# 目标高速
TARGET_ROUTE = 99

# 输出目录
OUTPUT_DIR = "./output/coordinate_correction"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("配置完成！")

## 1. 加载 Caltrans 官方数据

In [ ]:
print(f"加载 Caltrans 数据: {CALTRANS_GEOJSON}")
print("文件较大，请稍候...")

with open(CALTRANS_GEOJSON, 'r') as f:
    data = json.load(f)

print(f"总特征数: {len(data['features'])}")

# 只提取目标高速
records = []
for feature in tqdm(data['features'], desc="筛选数据"):
    props = feature.get('properties', {})
    if props.get('Route') != TARGET_ROUTE:
        continue
    
    geom = feature.get('geometry', {})
    coords = geom.get('coordinates', [None, None])
    
    odometer = props.get('Odometer')
    if odometer is None or coords[0] is None:
        continue
    
    records.append({
        'Route': props.get('Route'),
        'County': props.get('County'),
        'PM': props.get('PM'),
        'Odometer': odometer,
        'AlignCode': props.get('AlignCode'),
        'Direction': props.get('Direction'),
        'Longitude': coords[0],
        'Latitude': coords[1],
    })

caltrans_df = pd.DataFrame(records)
print(f"\nRoute {TARGET_ROUTE} 有效里程桩数: {len(caltrans_df)}")
print(f"\nAlignCode 分布:")
print(caltrans_df['AlignCode'].value_counts())
print(f"\nOdometer 范围: {caltrans_df['Odometer'].min():.2f} - {caltrans_df['Odometer'].max():.2f}")

## 2. 加载 PeMS 元数据

In [ ]:
META_COLUMNS = [
    'ID', 'Fwy', 'Dir', 'District', 'County', 'City',
    'State_PM', 'Abs_PM', 'Latitude', 'Longitude', 'Length',
    'Type', 'Lanes', 'Name', 'User_ID_1', 'User_ID_2',
    'User_ID_3', 'User_ID_4'
]

# 加载多个 district 的元数据
all_meta = []
for district in ['d03', 'd04', 'd05', 'd06', 'd07', 'd08', 'd10', 'd11', 'd12']:
    pattern = os.path.join(META_DIR, f"{district}_text_meta_*.txt")
    files = glob.glob(pattern)
    if files:
        meta_file = sorted(files)[-1]
        df = pd.read_csv(meta_file, sep='\t', names=META_COLUMNS, header=0, 
                         dtype={'ID': str, 'Fwy': str})
        all_meta.append(df)
        print(f"{district}: {len(df)} 条")

pems_df = pd.concat(all_meta, ignore_index=True)
print(f"\n总计: {len(pems_df)} 条")

In [ ]:
# 筛选目标高速
pems_target = pems_df[pems_df['Fwy'] == str(TARGET_ROUTE)].copy()
pems_target = pems_target[pems_target['Latitude'].notna() & pems_target['Abs_PM'].notna()]

print(f"PeMS {TARGET_ROUTE} 号高速站点: {len(pems_target)} 个")
print(f"\n方向分布:")
print(pems_target['Dir'].value_counts())
print(f"\n类型分布:")
print(pems_target['Type'].value_counts())
print(f"\nAbs_PM 范围: {pems_target['Abs_PM'].min():.2f} - {pems_target['Abs_PM'].max():.2f}")

## 3. 基于 Abs_PM 匹配修正坐标

In [ ]:
def calc_distance_feet(lat1, lon1, lat2, lon2):
    """计算两点距离（英尺）"""
    R = 3958.8 * 5280
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c


def correct_coordinates_by_abspm(pems_df, caltrans_df):
    """
    根据 Abs_PM 匹配 Caltrans Odometer，修正 PeMS 坐标
    
    匹配规则:
    - PeMS Dir N/E -> Caltrans AlignCode Right/Right Side
    - PeMS Dir S/W -> Caltrans AlignCode Left/Left Side
    - 在同方向数据中找 Odometer 最接近 Abs_PM 的点
    """
    results = []
    
    for _, row in tqdm(pems_df.iterrows(), total=len(pems_df), desc="修正坐标"):
        result = row.to_dict()
        
        # 保存原始坐标
        result['Original_Lat'] = row['Latitude']
        result['Original_Lon'] = row['Longitude']
        
        # 方向映射: N/E -> Right, S/W -> Left
        if row['Dir'] in ['N', 'E']:
            align_codes = ['Right', 'Right Side']
        else:
            align_codes = ['Left', 'Left Side']
        
        # 筛选同方向的 Caltrans 数据
        caltrans_subset = caltrans_df[
            caltrans_df['AlignCode'].isin(align_codes)
        ]
        
        if len(caltrans_subset) == 0:
            result['Corrected_Lat'] = None
            result['Corrected_Lon'] = None
            result['Match_Odometer'] = None
            result['PM_Diff'] = None
            result['Correction_Dist_ft'] = None
            result['Status'] = 'no_caltrans_data'
            results.append(result)
            continue
        
        # 找 Odometer 最接近 Abs_PM 的点
        pm_diff = (caltrans_subset['Odometer'] - row['Abs_PM']).abs()
        nearest_idx = pm_diff.idxmin()
        nearest = caltrans_subset.loc[nearest_idx]
        
        # 计算修正距离
        correction_dist = calc_distance_feet(
            row['Latitude'], row['Longitude'],
            nearest['Latitude'], nearest['Longitude']
        )
        
        result['Corrected_Lat'] = nearest['Latitude']
        result['Corrected_Lon'] = nearest['Longitude']
        result['Match_Odometer'] = nearest['Odometer']
        result['Match_AlignCode'] = nearest['AlignCode']
        result['PM_Diff'] = abs(row['Abs_PM'] - nearest['Odometer'])
        result['Correction_Dist_ft'] = correction_dist
        result['Status'] = 'corrected'
        
        results.append(result)
    
    return pd.DataFrame(results)


print("修正函数定义完成")

In [ ]:
# 执行修正
pems_corrected = correct_coordinates_by_abspm(pems_target, caltrans_df)

print("\n修正状态统计:")
print(pems_corrected['Status'].value_counts())

In [ ]:
# 修正距离统计
corrected = pems_corrected[pems_corrected['Status'] == 'corrected']

print("修正距离统计 (英尺):")
print(f"  平均: {corrected['Correction_Dist_ft'].mean():.1f}")
print(f"  中位数: {corrected['Correction_Dist_ft'].median():.1f}")
print(f"  最小: {corrected['Correction_Dist_ft'].min():.1f}")
print(f"  最大: {corrected['Correction_Dist_ft'].max():.1f}")

print("\n修正距离分布:")
bins = [(0, 50), (50, 100), (100, 200), (200, 500), (500, 1000), (1000, float('inf'))]
for low, high in bins:
    count = ((corrected['Correction_Dist_ft'] >= low) & (corrected['Correction_Dist_ft'] < high)).sum()
    label = f"{low}-{high}" if high != float('inf') else f">{low}"
    print(f"  {label} ft: {count} 个站点")

print("\nPM 匹配差异统计:")
print(f"  平均 PM 差异: {corrected['PM_Diff'].mean():.4f}")
print(f"  最大 PM 差异: {corrected['PM_Diff'].max():.4f}")

In [ ]:
# 显示修正距离最大的站点
print("修正距离最大的 15 个站点:")
top15 = corrected.nlargest(15, 'Correction_Dist_ft')[
    ['ID', 'Dir', 'Type', 'Abs_PM', 'Match_Odometer', 'PM_Diff', 'Correction_Dist_ft']
].copy()
top15['Correction_Dist_ft'] = top15['Correction_Dist_ft'].round(1)
print(top15.to_string(index=False))

In [ ]:
# 保存修正结果
output_file = os.path.join(OUTPUT_DIR, f'pems_{TARGET_ROUTE}_corrected.csv')
pems_corrected.to_csv(output_file, index=False)
print(f"已保存: {output_file}")

## 4. 可视化对比

In [ ]:
def create_comparison_map(pems_corrected, caltrans_df, output_path):
    """
    创建修正前后对比地图
    
    显示:
    - 原始 PeMS 坐标（红色）
    - 修正后坐标（绿色）
    - 修正连线（黄色虚线）
    - Caltrans 里程点（小蓝点，作为参考）
    """
    # 计算中心
    center_lat = pems_corrected['Original_Lat'].mean()
    center_lon = pems_corrected['Original_Lon'].mean()
    
    m = folium.Map(location=[center_lat, center_lon], zoom_start=9, tiles=None)
    
    # 添加底图
    folium.TileLayer('OpenStreetMap', name='OSM').add_to(m)
    folium.TileLayer(
        tiles='https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}',
        attr='Google', name='Google 街道'
    ).add_to(m)
    folium.TileLayer(
        tiles='https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
        attr='Google', name='Google 混合'
    ).add_to(m)
    
    # ========== Caltrans 里程点（参考）==========
    caltrans_group = folium.FeatureGroup(name='Caltrans 里程点', show=False)
    for _, row in caltrans_df.iterrows():
        color = '#2196F3' if row['AlignCode'] in ['Right', 'Right Side'] else '#00BCD4'
        folium.CircleMarker(
            [row['Latitude'], row['Longitude']],
            radius=2,
            color=color,
            fill=True,
            fillOpacity=0.6,
            weight=1,
            popup=f"Odometer={row['Odometer']:.2f}<br>AlignCode={row['AlignCode']}"
        ).add_to(caltrans_group)
    caltrans_group.add_to(m)
    
    # ========== 原始 PeMS 坐标（红色）==========
    original_group = folium.FeatureGroup(name='原始 PeMS 坐标')
    for _, row in pems_corrected.iterrows():
        folium.CircleMarker(
            [row['Original_Lat'], row['Original_Lon']],
            radius=6,
            color='#F44336',
            fill=True,
            fillColor='#F44336',
            fillOpacity=0.8,
            weight=2,
            popup=f"<b>原始位置</b><br>ID: {row['ID']}<br>Type: {row['Type']}<br>Dir: {row['Dir']}<br>Abs_PM: {row['Abs_PM']:.2f}",
            tooltip=f"原始 {row['ID']}"
        ).add_to(original_group)
    original_group.add_to(m)
    
    # ========== 修正后坐标（绿色）==========
    corrected_group = folium.FeatureGroup(name='修正后坐标')
    for _, row in pems_corrected.iterrows():
        if row['Status'] != 'corrected':
            continue
        folium.CircleMarker(
            [row['Corrected_Lat'], row['Corrected_Lon']],
            radius=6,
            color='#4CAF50',
            fill=True,
            fillColor='#4CAF50',
            fillOpacity=0.8,
            weight=2,
            popup=f"<b>修正后位置</b><br>ID: {row['ID']}<br>Type: {row['Type']}<br>偏移: {row['Correction_Dist_ft']:.0f} ft",
            tooltip=f"修正 {row['ID']} ({row['Correction_Dist_ft']:.0f}ft)"
        ).add_to(corrected_group)
    corrected_group.add_to(m)
    
    # ========== 修正连线（黄色虚线）==========
    lines_group = folium.FeatureGroup(name='修正连线')
    for _, row in pems_corrected.iterrows():
        if row['Status'] != 'corrected':
            continue
        if row['Correction_Dist_ft'] < 10:  # 太近的不画线
            continue
        
        # 根据偏移距离设置颜色
        if row['Correction_Dist_ft'] < 100:
            color = '#FFC107'  # 黄色
        elif row['Correction_Dist_ft'] < 500:
            color = '#FF9800'  # 橙色
        else:
            color = '#F44336'  # 红色
        
        folium.PolyLine(
            [[row['Original_Lat'], row['Original_Lon']],
             [row['Corrected_Lat'], row['Corrected_Lon']]],
            color=color,
            weight=2,
            dash_array='5,5',
            opacity=0.8,
            popup=f"{row['ID']}: {row['Correction_Dist_ft']:.0f} ft"
        ).add_to(lines_group)
    lines_group.add_to(m)
    
    # 图例
    legend = """
    <div style="position:fixed; bottom:50px; left:50px; z-index:1000;
                background:white; padding:12px; border:2px solid #333; border-radius:5px;
                font-size:12px;">
        <div style="font-weight:bold; margin-bottom:8px; font-size:14px;">坐标修正对比</div>
        <div><span style="color:#F44336;">●</span> 原始 PeMS 坐标</div>
        <div><span style="color:#4CAF50;">●</span> 修正后坐标</div>
        <div style="margin-top:5px;"><b>修正连线:</b></div>
        <div><span style="color:#FFC107;">---</span> < 100 ft</div>
        <div><span style="color:#FF9800;">---</span> 100-500 ft</div>
        <div><span style="color:#F44336;">---</span> > 500 ft</div>
        <div style="margin-top:5px;"><span style="color:#2196F3;">·</span> Caltrans Right</div>
        <div><span style="color:#00BCD4;">·</span> Caltrans Left</div>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend))
    
    folium.LayerControl().add_to(m)
    m.save(output_path)
    print(f"地图已保存: {output_path}")
    return m


print("可视化函数定义完成")

In [ ]:
# 生成对比地图
comparison_map = create_comparison_map(
    pems_corrected,
    caltrans_df,
    os.path.join(OUTPUT_DIR, f'correction_comparison_{TARGET_ROUTE}.html')
)

comparison_map

## 5. 按方向分开显示

In [ ]:
def create_direction_map(pems_corrected, caltrans_df, direction, output_path):
    """
    为单个方向创建详细的对比地图
    """
    # 筛选方向
    pems_dir = pems_corrected[pems_corrected['Dir'] == direction]
    
    if direction in ['N', 'E']:
        align_codes = ['Right', 'Right Side']
    else:
        align_codes = ['Left', 'Left Side']
    caltrans_dir = caltrans_df[caltrans_df['AlignCode'].isin(align_codes)]
    
    if len(pems_dir) == 0:
        print(f"无 {direction} 方向数据")
        return None
    
    center_lat = pems_dir['Original_Lat'].mean()
    center_lon = pems_dir['Original_Lon'].mean()
    
    m = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles=None)
    
    # 底图
    folium.TileLayer('OpenStreetMap', name='OSM').add_to(m)
    folium.TileLayer(
        tiles='https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
        attr='Google', name='Google 混合'
    ).add_to(m)
    
    # Caltrans 里程点
    caltrans_group = folium.FeatureGroup(name=f'Caltrans {direction}')
    for _, row in caltrans_dir.iterrows():
        folium.CircleMarker(
            [row['Latitude'], row['Longitude']],
            radius=3,
            color='#2196F3',
            fill=True,
            fillOpacity=0.7,
            weight=1,
            popup=f"Odometer={row['Odometer']:.2f}"
        ).add_to(caltrans_group)
    caltrans_group.add_to(m)
    
    # 站点类型颜色
    type_colors = {
        'ML': '#E53935',
        'HV': '#8E24AA',
        'OR': '#FF9800',
        'FR': '#795548',
        'FF': '#FFC107',
    }
    
    # 原始位置
    original_group = folium.FeatureGroup(name='原始位置')
    for _, row in pems_dir.iterrows():
        color = type_colors.get(row['Type'], '#888')
        folium.CircleMarker(
            [row['Original_Lat'], row['Original_Lon']],
            radius=8,
            color='#333',
            weight=2,
            fill=True,
            fillColor=color,
            fillOpacity=0.9,
            popup=f"<b>原始</b><br>{row['ID']}<br>{row['Type']}<br>PM={row['Abs_PM']:.2f}",
            tooltip=f"{row['ID']} ({row['Type']})"
        ).add_to(original_group)
    original_group.add_to(m)
    
    # 修正后位置
    corrected_group = folium.FeatureGroup(name='修正后位置')
    for _, row in pems_dir.iterrows():
        if row['Status'] != 'corrected':
            continue
        color = type_colors.get(row['Type'], '#888')
        folium.CircleMarker(
            [row['Corrected_Lat'], row['Corrected_Lon']],
            radius=8,
            color='#4CAF50',
            weight=3,
            fill=True,
            fillColor=color,
            fillOpacity=0.9,
            popup=f"<b>修正后</b><br>{row['ID']}<br>偏移={row['Correction_Dist_ft']:.0f}ft",
            tooltip=f"{row['ID']} 修正 {row['Correction_Dist_ft']:.0f}ft"
        ).add_to(corrected_group)
    corrected_group.add_to(m)
    
    # 连线
    lines_group = folium.FeatureGroup(name='修正连线')
    for _, row in pems_dir.iterrows():
        if row['Status'] != 'corrected' or row['Correction_Dist_ft'] < 10:
            continue
        folium.PolyLine(
            [[row['Original_Lat'], row['Original_Lon']],
             [row['Corrected_Lat'], row['Corrected_Lon']]],
            color='#FF5722',
            weight=2,
            dash_array='5,5'
        ).add_to(lines_group)
    lines_group.add_to(m)
    
    folium.LayerControl().add_to(m)
    m.save(output_path)
    print(f"已保存: {output_path}")
    return m


# 生成各方向地图
for direction in ['N', 'S']:
    create_direction_map(
        pems_corrected,
        caltrans_df,
        direction,
        os.path.join(OUTPUT_DIR, f'correction_{TARGET_ROUTE}{direction}.html')
    )

## 6. 导出修正后的元数据（用于后续构图）

In [ ]:
# 生成可用于构图的元数据
# 用修正后的坐标替换原始坐标

pems_for_graph = pems_corrected[pems_corrected['Status'] == 'corrected'].copy()

# 使用修正后的坐标
pems_for_graph['Latitude'] = pems_for_graph['Corrected_Lat']
pems_for_graph['Longitude'] = pems_for_graph['Corrected_Lon']

# 只保留构图需要的列
graph_columns = ['ID', 'Fwy', 'Dir', 'District', 'County', 'City',
                 'State_PM', 'Abs_PM', 'Latitude', 'Longitude', 'Length',
                 'Type', 'Lanes', 'Name']

available_cols = [c for c in graph_columns if c in pems_for_graph.columns]
pems_export = pems_for_graph[available_cols].copy()

# 保存
export_file = os.path.join(OUTPUT_DIR, f'pems_{TARGET_ROUTE}_meta_corrected.csv')
pems_export.to_csv(export_file, index=False)
print(f"已保存修正后元数据: {export_file}")
print(f"站点数: {len(pems_export)}")

## 总结

### 输出文件

| 文件 | 说明 |
|------|------|
| `pems_99_corrected.csv` | 完整修正结果，包含原始和修正坐标 |
| `pems_99_meta_corrected.csv` | 修正后的元数据，可直接用于构图 |
| `correction_comparison_99.html` | 修正前后对比地图 |
| `correction_99N.html` | N方向详细地图 |
| `correction_99S.html` | S方向详细地图 |

### 在分层构图中使用

```python
# 加载修正后的元数据
meta_df = pd.read_csv('./output/coordinate_correction/pems_99_meta_corrected.csv')

# 坐标已修正，OSRM 会匹配到正确的道路
```